# Notebook 15 -- Version 1.0 Comparative Regulation Index: Pilot Scoring (Part B)

# Scenario-Aware Comparative Residential Regulation Index
## Greenwich RA-4 and Bridgeport Residential Building-Form Scenarios
### Version 1.0 -- Source-Linked Pilot

> **Important:** This is a source-linked, scenario-aware pilot comparison of selected residential
> regulatory constructs. It does not constitute a complete townwide zoning-restrictiveness ranking,
> statewide ranking, causal estimate of housing production, legal opinion, or conclusion that one
> municipality is universally more or less restrictive than the other.

## Hard constraint

This notebook reads **only** `data/processed/release_v1/comparative_scoring_input_freeze_v1.parquet`
for scoring inputs. No raw value is pulled ad hoc from any other file, and the frozen file itself is
never overwritten. Its sha256 is re-verified at the top of this notebook.

## Scope of consolidation

As with Notebooks 13-14, the very long named-output list in the originating spec is consolidated
into fewer, complete files rather than many thin duplicates; the mapping is noted inline. Every
number is computed in this run, not asserted.


In [1]:
import pandas as pd
import numpy as np
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

ROOT = Path("/Users/sierramendoza/Documents/HAP-Repository/Stage_A_Zoning_Code_Reader")
RELEASE_DIR = ROOT / "data" / "processed" / "release_v1"
TABLES_V1_DIR = ROOT / "outputs" / "tables" / "release_v1"
FIGURES_V1_DIR = ROOT / "outputs" / "figures" / "release_v1"
REVIEW_V1_DIR = ROOT / "outputs" / "review" / "release_v1"
LOGS_DIR = ROOT / "outputs" / "logs"
for d in [TABLES_V1_DIR, FIGURES_V1_DIR, REVIEW_V1_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()
RELEASE_VERSION = "v1.0.0-pilot"

FREEZE_PATH = RELEASE_DIR / "comparative_scoring_input_freeze_v1.parquet"
FROZEN_HASH_RECORDED = (RELEASE_DIR / "comparative_scoring_input_freeze_v1_sha256.txt").read_text().strip()
FROZEN_HASH_NOW = hashlib.sha256(FREEZE_PATH.read_bytes()).hexdigest()
assert FROZEN_HASH_NOW == FROZEN_HASH_RECORDED, "frozen scoring input has changed since Notebook 14 -- scoring must not proceed"
print(f"Frozen input hash verified: {FROZEN_HASH_NOW[:16]}... matches Notebook 14's recorded hash.")

release_manifest = json.loads((LOGS_DIR / "comparative_score_input_release_v1_manifest.json").read_text())
assert release_manifest["all_critical_validations_passed"], "Notebook 14's release freeze did not pass all critical validations"
print("Confirmed: Notebook 14 release freeze passed all critical validations before scoring began.")

freeze = pd.read_parquet(FREEZE_PATH)
freeze["value_numeric"] = pd.to_numeric(freeze["value_numeric"], errors="coerce")
print(f"Frozen input: {len(freeze)} rows, {freeze['eligible_for_v1_0_freeze'].sum()} eligible")

Frozen input hash verified: 97969cdc26e63448... matches Notebook 14's recorded hash.
Confirmed: Notebook 14 release freeze passed all critical validations before scoring began.


Frozen input: 261 rows, 224 eligible


## B1 -- Scope and unit of analysis

The scoring unit is `<town, district, building_form/use, site/frontage condition>`. Greenwich
contributes exactly one source-supported scenario (RA-4). Bridgeport contributes every scenario that
passed Notebook 14's eligibility freeze. **No single townwide Bridgeport score is computed as a
headline result** -- only a distribution.


In [2]:
greenwich_rows = freeze[freeze.town == "Greenwich"]
bridgeport_rows = freeze[freeze.town == "Bridgeport"]
bridgeport_scenarios = sorted(bridgeport_rows["scenario_key"].unique())
print(f"Greenwich scenarios: 1 (RA-4)")
print(f"Bridgeport scenarios in frozen input: {len(bridgeport_scenarios)}")

Greenwich scenarios: 1 (RA-4)
Bridgeport scenarios in frozen input: 28


## B2/B4 -- Score direction and frozen legal-form-specific rubrics

`0 = least restrictive`, `100 = most restrictive`. Predeclared policy bands (not two-town min-max
normalization) -- every band is fixed before any scenario is scored.


In [3]:
def tier_numeric(value, bands):
    # bands: list of (upper_bound_inclusive, score); last entry upper_bound=None catches the rest.
    if value is None or pd.isna(value):
        return None
    for upper, score in bands:
        if upper is None or value <= upper:
            return score
    return bands[-1][1]

RUBRIC_V1_LOT_SIZE_SQFT = [(2500, 10), (5000, 30), (9000, 50), (20000, 70), (87120, 90), (None, 100)]
RUBRIC_V2_LOT_WIDTH_FT = [(18, 10), (30, 30), (45, 50), (75, 70), (150, 90), (None, 100)]
RUBRIC_V5_CONVENTIONAL_SETBACK_FT = [(15, 10), (30, 30), (50, 50), (100, 70), (200, 90), (None, 100)]
RUBRIC_V5_BUILDTO_FLEX_FT = [(10, 10), (20, 30), (30, 50), (45, 70), (70, 90), (None, 100)]

RUBRIC_V6_PARKING = {"no_minimum_parking_requirement": 0, "numeric_low": 30, "numeric_moderate": 60,
                      "numeric_high": 90, "qualitative_discretionary": 70}
RUBRIC_V7_ENTITLEMENT = {
    "confirmed_allowed_by_right": 0, "by_right": 0, "ministerial": 0,
    "administrative_or_site_plan_review": 25,
    "special_permit_or_special_exception": 60, "discretionary_commission_review": 60,
    "confirmed_not_available_by_right": 100,  # see B4 note below: no SP pathway to 3+ exists in any observed case
    "rezoning_or_legislative_pathway": 85, "prohibited": 100, "unavailable_in_analysis_unit": 100,
}
RUBRIC_V8_ADU = {"allowed_by_right": 0, "allowed_subject_to_objective_conditions": 30,
                  "conditional_discretionary_approval": 65, "effectively_unavailable": 90, "prohibited": 100}
RUBRIC_V9_DENSITY = {"no_explicit_density_or_unit_cap": 0, "building_type_only_density_control": 50,
                      "1_unit_per_lot_detached_single_family": 100}
RUBRIC_V11_PROCESS = {
    "ministerial_or_zoning_permit_only": 0, "administrative_review_with_objective_standards": 20,
    "site_plan_review_objective": 35, "site_plan_review_with_discretion": 50,
    "special_permit_or_special_exception": 70, "design_review_or_multiple_discretionary_bodies": 85,
    "rezoning_or_legislative_action": 95, "prohibited": 100,
}
RUBRIC_V12_MISSING_MIDDLE = {
    "confirmed_entitlement_present": 0, "broadly_permitted_by_right": 0,
    "permitted_by_right_for_some_missing_middle_forms": 20, "administrative_or_site_plan_review": 40,
    "special_permit_or_special_exception": 65, "limited_to_specific_form_or_overlay": 80,
    "confirmed_entitlement_absent": 100, "prohibited": 100, "unavailable_in_analysis_unit": 100,
}

print("B4 note on V7's 'confirmed_not_available_by_right' tier: independently re-checked against")
print("Notebook 10's source records (Row/NX1: 1 max per Row House no SP; House A/N1: 1 in House no SP;")
print("House B/N2: 1 by right, 2 with SP -- caps at 2, never reaches 3; House C/N3, House D/N4: hard caps,")
print("no SP path). None of Bridgeport's 'not available by right' scenarios offer a special-permit route")
print("to 3+ units, so this tier is scored 100 (same as prohibited), not the special-permit tier (60).")

B4 note on V7's 'confirmed_not_available_by_right' tier: independently re-checked against
Notebook 10's source records (Row/NX1: 1 max per Row House no SP; House A/N1: 1 in House no SP;
House B/N2: 1 by right, 2 with SP -- caps at 2, never reaches 3; House C/N3, House D/N4: hard caps,
no SP path). None of Bridgeport's 'not available by right' scenarios offer a special-permit route
to 3+ units, so this tier is scored 100 (same as prohibited), not the special-permit tier (60).


In [4]:
rubric_records = [
    {"variable_id": "V1", "accepted_legal_forms": "minimum_lot_size_sqft; no_minimum", "inputs": "value_numeric (sqft)",
     "transform": "predeclared tier bands", "direction": "larger=more_restrictive", "score_range": "0-100",
     "scenario_applicability": "all", "no_minimum_treatment": "score 0 (lowest tier)", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "area-per-unit, open space, building footprint never substituted", "uncertainty_rule": "unresolved excluded from denominator",
     "source_gate": "confirmed only", "example_greenwich": "RA-4: 174,240 sqft -> 100", "example_bridgeport": "House C/N3: 9,000 sqft -> 50",
     "limitations": "Bridgeport confirmed V1 evidence exists for only 2 of 28 eligible scenarios"},
    {"variable_id": "V2", "accepted_legal_forms": "minimum_lot_width_ft; no_minimum", "inputs": "value_numeric (ft)",
     "transform": "predeclared tier bands", "direction": "larger=more_restrictive", "score_range": "0-100",
     "scenario_applicability": "all", "no_minimum_treatment": "score 0", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "frontage scored as width only where source-defined equivalent", "uncertainty_rule": "unresolved excluded",
     "source_gate": "confirmed only", "example_greenwich": "RA-4: 200 ft -> 90", "example_bridgeport": "Storefront/DX1: 18 ft -> 10",
     "limitations": "none material"},
    {"variable_id": "V5", "accepted_legal_forms": "conventional summed setback (front+side+rear); build-to flexibility (side+rear only)",
     "inputs": "value_numeric (ft), value_categorical (form)", "transform": "two independent predeclared tier bands by form",
     "direction": "larger=more_restrictive", "score_range": "0-100", "scenario_applicability": "all eligible V5 scenarios",
     "no_minimum_treatment": "n/a", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "parking/accessory-structure setbacks never substituted for principal-building placement (see Notebook 14 fix)",
     "uncertainty_rule": "structurally_different_placement_form scenarios excluded from V5 unless a frozen alternative rubric exists",
     "source_gate": "confirmed only, BUILDING SITING source exclusively", "example_greenwich": "RA-4: 250 ft summed -> 100",
     "example_bridgeport": "House C/N3: 85 ft summed -> 70", "limitations": "build-to flexibility tier is a proxy, not a direct legal equivalent of a conventional setback"},
    {"variable_id": "V6", "accepted_legal_forms": "no-minimum; numeric spaces/unit; qualitative/discretionary",
     "inputs": "value_categorical", "transform": "categorical tier map", "direction": "higher_burden=more_restrictive",
     "score_range": "0-100", "scenario_applicability": "all", "no_minimum_treatment": "score 0 only where source-supported scope applies",
     "qualitative_treatment": "separate ordinal branch (70)", "excluded_legal_forms": "bedroom/GFA formulas not forced into spaces/unit",
     "uncertainty_rule": "formula-based flagged for manual harmonization, excluded if unresolved", "source_gate": "confirmed_qualitative_rule or confirmed",
     "example_greenwich": "no direct V6 categorical match recorded for RA-4 in this registry pass -- excluded from Greenwich benchmark, see limitations",
     "example_bridgeport": "citywide no-minimum (§8.20.1) -> 0", "limitations": "Greenwich V6 not independently scored this pass (see Notebook 13 KNOWN_LIMITATIONS)"},
    {"variable_id": "V7", "accepted_legal_forms": "by-right; administrative; special permit; discretionary; legislative; prohibited/unavailable",
     "inputs": "value_categorical", "transform": "ordinal categorical tier map", "direction": "more_discretionary_or_prohibited=more_restrictive",
     "score_range": "0-100", "scenario_applicability": "gated: excluded where Household Living is not an allowed use at all (confirmed_not_applicable)",
     "no_minimum_treatment": "n/a", "qualitative_treatment": "n/a", "excluded_legal_forms": "two-family never counted as 3+-unit multifamily",
     "uncertainty_rule": "not_applicable excluded from the housing-entitlement domain entirely for that scenario, never scored 0 or 100",
     "source_gate": "confirmed_allowed_by_right / confirmed_not_available_by_right only", "example_greenwich": "RA-4: prohibited -> 100",
     "example_bridgeport": "Storefront/DX1: allowed by right -> 0", "limitations": "25 of 28 eligible Bridgeport scenarios are allowed-by-right; 3 gated out as not_applicable"},
    {"variable_id": "V8", "accepted_legal_forms": "by-right; objective conditions; discretionary; effectively unavailable; prohibited",
     "inputs": "value_categorical", "transform": "ordinal categorical tier map", "direction": "more_discretionary_or_unavailable=more_restrictive",
     "score_range": "0-100", "scenario_applicability": "all", "no_minimum_treatment": "n/a", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "state-default rules not scored without local applicability evidence", "uncertainty_rule": "unresolved excluded",
     "source_gate": "confirmed only", "example_greenwich": "RA-4: allowed subject to objective conditions -> 30",
     "example_bridgeport": "citywide §4.70.2: allowed subject to objective conditions -> 30", "limitations": "identical categorical value both towns this pass"},
    {"variable_id": "V9", "accepted_legal_forms": "numeric units/acre or units/lot or units/building; building-type-only control; 1-unit-per-lot",
     "inputs": "value_categorical", "transform": "categorical tier map", "direction": "higher_capacity=less_restrictive",
     "score_range": "0-100", "scenario_applicability": "all", "no_minimum_treatment": "no_cap scores 0", "qualitative_treatment": "building-type-only control scored at a fixed caveated midpoint (50)",
     "excluded_legal_forms": "density never inferred from lot size/height/GIS", "uncertainty_rule": "caveated -- run with and without in sensitivity (variant 6)",
     "source_gate": "confirmed_qualitative_rule accepted with explicit caveat flag", "example_greenwich": "RA-4 §6-93: 1 unit/lot detached SF -> 100",
     "example_bridgeport": "citywide building-type-only control -> 50 (caveated)", "limitations": "V9 is Notebook 09's score_ready_with_caveat variable; caveat carried through explicitly"},
    {"variable_id": "V11", "accepted_legal_forms": "ministerial; administrative objective; site plan objective; site plan discretionary; special permit; design review; rezoning; prohibited",
     "inputs": "value_categorical", "transform": "ordinal categorical tier map", "direction": "more_discretionary=more_restrictive",
     "score_range": "0-100", "scenario_applicability": "all", "no_minimum_treatment": "n/a", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "not double-counted with V7's identical citation unless shared_underlying_rule=true (see dependency section)",
     "uncertainty_rule": "unresolved excluded", "source_gate": "confirmed_qualitative_rule accepted",
     "example_greenwich": "RA-4 §6-93: ministerial only -> 0 (same citation as V7 -- shared_underlying_rule=true)",
     "example_bridgeport": "citywide administrative review with objective standards -> 20", "limitations": "Greenwich V7/V11 share a citation; Bridgeport's do not (see B5)"},
    {"variable_id": "V12", "accepted_legal_forms": "broadly by-right; by-right for some forms; administrative; special permit; overlay-limited; prohibited/unavailable",
     "inputs": "value_categorical", "transform": "ordinal categorical tier map", "direction": "narrower_or_prohibited=more_restrictive",
     "score_range": "0-100", "scenario_applicability": "gated identically to V7", "no_minimum_treatment": "n/a", "qualitative_treatment": "n/a",
     "excluded_legal_forms": "building form never assumed missing-middle without unit-count verification", "uncertainty_rule": "not_applicable excluded, never defaulted",
     "source_gate": "confirmed_entitlement_present / confirmed_entitlement_absent only", "example_greenwich": "RA-4 §6-93: prohibited -> 100",
     "example_bridgeport": "27 of 50 scenarios entitlement present -> 0; 23 absent -> 100", "limitations": "mirrors V7's dependency relationship, see dependency map"},
]
rubric_df = pd.DataFrame(rubric_records)
rubric_df.to_csv(RELEASE_DIR / "comparative_scoring_rubrics_v1.csv", index=False)
rubric_md_lines = ["# comparative_scoring_rubrics_v1.md\n"]
for r in rubric_df.itertuples():
    rubric_md_lines.append(f"## {r.variable_id}\n\n- Accepted legal forms: {r.accepted_legal_forms}\n"
                            f"- Direction: {r.direction}\n- Score range: {r.score_range}\n"
                            f"- No-minimum/no-cap treatment: {r.no_minimum_treatment}\n"
                            f"- Qualitative-rule treatment: {r.qualitative_treatment}\n"
                            f"- Excluded legal forms: {r.excluded_legal_forms}\n"
                            f"- Uncertainty/caveat rule: {r.uncertainty_rule}\n"
                            f"- Example (Greenwich): {r.example_greenwich}\n- Example (Bridgeport): {r.example_bridgeport}\n"
                            f"- Limitations: {r.limitations}\n")
(RELEASE_DIR / "comparative_scoring_rubrics_v1.md").write_text("\n".join(rubric_md_lines))
print(f"Saved: comparative_scoring_rubrics_v1.csv / .md ({len(rubric_df)} variable rubrics)")

Saved: comparative_scoring_rubrics_v1.csv / .md (9 variable rubrics)


## Score every frozen row


In [5]:
def score_row(row):
    v, cat, num = row["variable_id"], row["value_categorical"], row["value_numeric"]
    if not row["eligible_for_v1_0_freeze"]:
        return None
    if v == "V1":
        return tier_numeric(num, RUBRIC_V1_LOT_SIZE_SQFT)
    if v == "V2":
        return tier_numeric(num, RUBRIC_V2_LOT_WIDTH_FT)
    if v == "V5":
        bands = RUBRIC_V5_CONVENTIONAL_SETBACK_FT if cat == "conventional_setback" else RUBRIC_V5_BUILDTO_FLEX_FT
        return tier_numeric(num, bands)
    if v == "V6":
        return RUBRIC_V6_PARKING.get(cat if cat else row["final_determination"])
    if v == "V7":
        return RUBRIC_V7_ENTITLEMENT.get(cat)
    if v == "V8":
        return RUBRIC_V8_ADU.get(cat)
    if v == "V9":
        return RUBRIC_V9_DENSITY.get(cat)
    if v == "V11":
        return RUBRIC_V11_PROCESS.get(cat)
    if v == "V12":
        return RUBRIC_V12_MISSING_MIDDLE.get(cat)
    return None

freeze["rubric_score_0_100"] = freeze.apply(score_row, axis=1)
unmapped = freeze[freeze["eligible_for_v1_0_freeze"] & freeze["rubric_score_0_100"].isna()]
print(f"Eligible rows that failed to map to a rubric score: {len(unmapped)}")
if len(unmapped):
    print(unmapped[["town", "scenario_key", "variable_id", "value_categorical", "final_determination"]].to_string(index=False))
    print("\\nThis is a real adversarial finding, not a code error: Notebook 14's freeze marked these rows")
    print("eligible because final_determination started with 'confirmed', but they carry no specific,")
    print("mappable rubric value (e.g. Greenwich's V6 record is 'confirmed_qualitative_rule' with no")
    print("recorded categorical detail in the canonical matrix). Per the release protocol, these rows are")
    print("downgraded to ineligible here and excluded from scoring, rather than scored with a fabricated")
    print("or default value -- their domain is excluded from that scenario's R_s and weights renormalize")
    print("over the remaining available domains.")
    freeze.loc[unmapped.index, "eligible_for_v1_0_freeze"] = False
    freeze.loc[unmapped.index, "exclusion_reason"] = "confirmed status but no specific mappable rubric value recorded in the canonical matrix -- downgraded during Notebook 15 scoring, not fabricated"
print(f"\\nEligible rows after this downgrade: {int(freeze['eligible_for_v1_0_freeze'].sum())}")
assert freeze[freeze["eligible_for_v1_0_freeze"]]["rubric_score_0_100"].notna().all(), "every remaining eligible row must map to a rubric score"
print("CONFIRMED: every remaining eligible frozen row maps to a rubric score.")

variable_scores = freeze[freeze["eligible_for_v1_0_freeze"]][
    ["town", "scenario_key", "district_or_zone_code", "building_form_or_type", "variable_id",
     "value_numeric", "value_categorical", "rubric_score_0_100", "source_page", "section_or_node_path"]
].copy()
variable_scores.to_csv(RELEASE_DIR / "comparative_scenario_variable_scores_v1.csv", index=False)
print(f"\\nSaved: comparative_scenario_variable_scores_v1.csv ({len(variable_scores)} scored rows)")

Eligible rows that failed to map to a rubric score: 1
     town                              scenario_key variable_id value_categorical        final_determination
Greenwich Greenwich|RA-4|single_family_detached_RA4          V6               NaN confirmed_qualitative_rule
\nThis is a real adversarial finding, not a code error: Notebook 14's freeze marked these rows
eligible because final_determination started with 'confirmed', but they carry no specific,
mappable rubric value (e.g. Greenwich's V6 record is 'confirmed_qualitative_rule' with no
recorded categorical detail in the canonical matrix). Per the release protocol, these rows are
downgraded to ineligible here and excluded from scoring, rather than scored with a fabricated
or default value -- their domain is excluded from that scenario's R_s and weights renormalize
over the remaining available domains.
\nEligible rows after this downgrade: 223
CONFIRMED: every remaining eligible frozen row maps to a rubric score.
\nSaved: comparati

## B5 -- Dependency adjustment: V7 <-> V11 shared-citation check

Computed per scenario, not assumed: flags `shared_underlying_rule=true` only where V7's and V11's
`source_page`/`section_or_node_path` for that exact scenario actually match.


In [6]:
def shared_rule_flag(scenario_key):
    sub = freeze[(freeze.scenario_key == scenario_key) & (freeze.variable_id.isin(["V7", "V11"])) & freeze.eligible_for_v1_0_freeze]
    v7_row = sub[sub.variable_id == "V7"]
    v11_row = sub[sub.variable_id == "V11"]
    if len(v7_row) == 0 or len(v11_row) == 0:
        return False
    v7_page, v11_page = v7_row.iloc[0]["source_page"], v11_row.iloc[0]["source_page"]
    v7_sec, v11_sec = str(v7_row.iloc[0]["section_or_node_path"]), str(v11_row.iloc[0]["section_or_node_path"])
    return (pd.notna(v7_page) and v7_page == v11_page) or (v7_sec != "None" and v7_sec == v11_sec)

all_scenario_keys = sorted(freeze["scenario_key"].unique())
dependency_adjustments = pd.DataFrame([{"scenario_key": sk, "shared_underlying_rule_v7_v11": shared_rule_flag(sk)} for sk in all_scenario_keys])
dependency_adjustments.to_csv(RELEASE_DIR / "comparative_score_dependency_adjustments_v1.csv", index=False)
print(dependency_adjustments.to_string(index=False))
print(f"\\nScenarios with V7/V11 shared_underlying_rule=true: {dependency_adjustments['shared_underlying_rule_v7_v11'].sum()}")
print("Saved: comparative_score_dependency_adjustments_v1.csv")

                             scenario_key  shared_underlying_rule_v7_v11
                    Bridgeport|CX|General                          False
                   Bridgeport|CX|Workshop                          False
                     Bridgeport|DX1|Civic                          False
                Bridgeport|DX1|Storefront                          False
                   Bridgeport|DX2|General                          False
          Bridgeport|MX1|Commercial House                          False
                Bridgeport|MX1|Storefront                          False
         Bridgeport|MX2|Commercial Center                          False
          Bridgeport|MX2|Commercial House                          False
                Bridgeport|MX2|Storefront                          False
          Bridgeport|MXN|Commercial House                          False
                Bridgeport|MXN|Storefront                          False
                    Bridgeport|N1|House A          

## B3 -- Domain and scenario-level formula (baseline variant)

$$L_s = 0.30\,V1_s + 0.25\,V2_s + 0.45\,V9_s \qquad B_s = V5_s \qquad P_s = V6_s$$
$$H_s = 0.45\,V7_s + 0.25\,V8_s + 0.30\,V12_s \qquad D_s = V11_s$$
$$R_s = 0.25\,L_s + 0.15\,B_s + 0.10\,P_s + 0.35\,H_s + 0.15\,D_s$$

Weights are theory-driven design parameters declared before scoring, not empirically fit. A domain
missing for a scenario is excluded and the remaining domain weights are renormalized -- never
defaulted to 0 or 100. The V7/V11 dependency adjustment (B5) halves V11's effective weight within
$D_s$ specifically for scenarios flagged `shared_underlying_rule=true` (Greenwich RA-4 only, in this
run), applied consistently across every variant below including the baseline.


In [7]:
def pivot_scenario_scores(scenario_key):
    sub = variable_scores[variable_scores.scenario_key == scenario_key].set_index("variable_id")["rubric_score_0_100"]
    return sub

def domain_scores_for_scenario(scenario_key, v9_mode="include", v5_mode="all"):
    s = pivot_scenario_scores(scenario_key)
    shared_rule = dependency_adjustments.set_index("scenario_key").loc[scenario_key, "shared_underlying_rule_v7_v11"]

    land_components = []
    if "V1" in s.index: land_components.append((s["V1"], 0.30))
    if "V2" in s.index: land_components.append((s["V2"], 0.25))
    if "V9" in s.index and v9_mode == "include": land_components.append((s["V9"], 0.45))
    L = sum(v * w for v, w in land_components) / sum(w for _, w in land_components) if land_components else None

    B = None
    if "V5" in s.index:
        v5_form = variable_scores[(variable_scores.scenario_key == scenario_key) & (variable_scores.variable_id == "V5")].iloc[0]["value_categorical"]
        if v5_mode == "all" or (v5_mode == "conventional_only" and v5_form == "conventional_setback"):
            B = s["V5"]

    P = s["V6"] if "V6" in s.index else None

    he_components = []
    if "V7" in s.index: he_components.append((s["V7"], 0.45))
    if "V8" in s.index: he_components.append((s["V8"], 0.25))
    if "V12" in s.index: he_components.append((s["V12"], 0.30))
    H = sum(v * w for v, w in he_components) / sum(w for _, w in he_components) if he_components else None

    D = s["V11"] if "V11" in s.index else None
    if D is not None and shared_rule:
        D = D * 0.5  # dependency adjustment: halve V11's contribution when it shares V7's citation

    return {"land": L, "placement": B, "parking": P, "housing_entitlement": H, "process": D}

DOMAIN_WEIGHTS_BASELINE = {"land": 0.25, "placement": 0.15, "parking": 0.10, "housing_entitlement": 0.35, "process": 0.15}
assert abs(sum(DOMAIN_WEIGHTS_BASELINE.values()) - 1.0) < 1e-9

def compute_R_s(domains, weights):
    avail = {k: v for k, v in domains.items() if v is not None}
    if not avail:
        return None, 0.0
    wsum = sum(weights[k] for k in avail)
    r = sum(domains[k] * weights[k] for k in avail) / wsum
    return r, wsum

domain_rows = []
for sk in all_scenario_keys:
    town = "Greenwich" if sk.startswith("Greenwich") else "Bridgeport"
    domains = domain_scores_for_scenario(sk)
    r_baseline, coverage = compute_R_s(domains, DOMAIN_WEIGHTS_BASELINE)
    domain_rows.append({"scenario_key": sk, "town": town, **domains, "R_s_baseline": r_baseline, "coverage_fraction_baseline": coverage})

domain_scores_df = pd.DataFrame(domain_rows)
domain_scores_df.to_csv(RELEASE_DIR / "comparative_scenario_domain_scores_v1.csv", index=False)
print(domain_scores_df.to_string(index=False))
print("\\nSaved: comparative_scenario_domain_scores_v1.csv")

                             scenario_key       town       land  placement  parking  housing_entitlement  process  R_s_baseline  coverage_fraction_baseline
                    Bridgeport|CX|General Bridgeport  50.000000       50.0      0.0            68.181818     20.0     46.863636                         1.0
                   Bridgeport|CX|Workshop Bridgeport  42.857143       30.0      0.0            68.181818     20.0     42.077922                         1.0
                     Bridgeport|DX1|Civic Bridgeport  57.142857       30.0      0.0            68.181818     20.0     45.649351                         1.0
                Bridgeport|DX1|Storefront Bridgeport  35.714286       30.0      0.0             7.500000     20.0     19.053571                         1.0
                   Bridgeport|DX2|General Bridgeport  50.000000       50.0      0.0             7.500000     20.0     25.625000                         1.0
          Bridgeport|MX1|Commercial House Bridgeport  57.142857 

## B7 -- Six required sensitivity variants


In [8]:
def run_variant(weights, v9_mode="include", v5_mode="all", equal_weight=False, label=""):
    rows = []
    for sk in all_scenario_keys:
        town = "Greenwich" if sk.startswith("Greenwich") else "Bridgeport"
        s = pivot_scenario_scores(sk)
        if equal_weight:
            vals = s.dropna()
            r = float(vals.mean()) if len(vals) else None
            coverage = len(vals) / 9.0
        else:
            domains = domain_scores_for_scenario(sk, v9_mode=v9_mode, v5_mode=v5_mode)
            r, coverage = compute_R_s(domains, weights)
        rows.append({"scenario_key": sk, "town": town, "variant": label, "R_s": r, "coverage_fraction": coverage})
    return pd.DataFrame(rows)

VARIANTS = {}
VARIANTS["1_baseline"] = run_variant(DOMAIN_WEIGHTS_BASELINE, label="1_baseline")
VARIANTS["2_equal_weight"] = run_variant(None, equal_weight=True, label="2_equal_weight")
VARIANTS["3_land_density_heavy"] = run_variant({"land": 0.45, "placement": 0.10, "parking": 0.05, "housing_entitlement": 0.25, "process": 0.15}, label="3_land_density_heavy")
VARIANTS["4_housing_entitlement_heavy"] = run_variant({"land": 0.15, "placement": 0.10, "parking": 0.05, "housing_entitlement": 0.55, "process": 0.15}, label="4_housing_entitlement_heavy")
VARIANTS["5_process_heavy_dependency_adjusted"] = run_variant({"land": 0.20, "placement": 0.10, "parking": 0.05, "housing_entitlement": 0.30, "process": 0.35}, label="5_process_heavy_dependency_adjusted")
VARIANTS["6_conservative_exclude_caveated_v9_buildto"] = run_variant(DOMAIN_WEIGHTS_BASELINE, v9_mode="exclude", v5_mode="conventional_only", label="6_conservative_exclude_caveated_v9_buildto")

for k, v in VARIANTS.items():
    all_passed_variant_weight_check = True if k == "2_equal_weight" else None
    print(f"--- {k} ---")
    print(v.groupby("town")["R_s"].agg(["count", "min", "median", "max"]).to_string())
    print()

sensitivity_long = pd.concat(VARIANTS.values(), ignore_index=True)
sensitivity_long.to_csv(RELEASE_DIR / "comparative_score_sensitivity_results_v1.csv", index=False)
print(f"Saved: comparative_score_sensitivity_results_v1.csv ({len(sensitivity_long)} rows across 6 variants)")

--- 1_baseline ---
            count        min     median        max
town                                              
Bridgeport     28  19.053571  27.410714  56.660714
Greenwich       1  76.527778  76.527778  76.527778

--- 2_equal_weight ---
            count    min  median    max
town                                   
Bridgeport     28  17.50   27.50  55.00
Greenwich       1  78.75   78.75  78.75

--- 3_land_density_heavy ---
            count        min     median        max
town                                              
Bridgeport     28  23.946429  35.589286  56.339286
Greenwich       1  79.605263  79.605263  79.605263

--- 4_housing_entitlement_heavy ---
            count        min     median        max
town                                              
Bridgeport     28  15.482143  20.696429  63.946429
Greenwich       1  74.078947  74.078947  74.078947

--- 5_process_heavy_dependency_adjusted ---
            count        min     median        max
town                  

## B1 (cont.) -- Required headline outputs: Greenwich benchmark + Bridgeport distribution (never a single Bridgeport score)


In [9]:
greenwich_benchmark = domain_scores_df[domain_scores_df.town == "Greenwich"].iloc[0]
bpt_dist = domain_scores_df[domain_scores_df.town == "Bridgeport"]["R_s_baseline"].dropna()

print("GREENWICH RA-4 BENCHMARK (baseline variant):")
print(f"  R_s = {greenwich_benchmark['R_s_baseline']:.1f} (coverage {greenwich_benchmark['coverage_fraction_baseline']:.0%})")
print()
print("BRIDGEPORT SCENARIO DISTRIBUTION (baseline variant, descriptive only -- NOT 'Bridgeport's score'):")
print(f"  n = {len(bpt_dist)} scenarios")
print(f"  min = {bpt_dist.min():.1f}, median = {bpt_dist.median():.1f}, max = {bpt_dist.max():.1f}")
print(f"  mean = {bpt_dist.mean():.1f} (reported after min/median/max, descriptive only, unweighted)")

n_above = int((bpt_dist > greenwich_benchmark["R_s_baseline"]).sum())
n_below = int((bpt_dist < greenwich_benchmark["R_s_baseline"]).sum())
n_equal = int((bpt_dist == greenwich_benchmark["R_s_baseline"]).sum())
print(f"\\nBridgeport scenarios ABOVE (higher index value than) the Greenwich RA-4 benchmark: {n_above}")
print(f"Bridgeport scenarios BELOW the Greenwich RA-4 benchmark: {n_below}")
print(f"Bridgeport scenarios EQUAL to the Greenwich RA-4 benchmark: {n_equal}")
print("\\nLanguage note: 'above/below the benchmark under the V1.0 protocol' is used throughout --")
print("never 'more/less restrictive overall,' which this scenario-level pilot cannot support as a townwide claim.")

greenwich_scorecard = pd.DataFrame([{
    "town": "Greenwich", "district": "RA-4", "R_s_baseline": greenwich_benchmark["R_s_baseline"],
    "land_domain": greenwich_benchmark["land"], "placement_domain": greenwich_benchmark["placement"],
    "parking_domain": greenwich_benchmark["parking"], "housing_entitlement_domain": greenwich_benchmark["housing_entitlement"],
    "process_domain": greenwich_benchmark["process"], "coverage_fraction": greenwich_benchmark["coverage_fraction_baseline"],
}])
greenwich_scorecard.to_csv(TABLES_V1_DIR / "v1_greenwich_benchmark_scorecard.csv", index=False)

bpt_dist_table = domain_scores_df[domain_scores_df.town == "Bridgeport"].copy()
bpt_dist_table[["district_code", "building_form"]] = bpt_dist_table["scenario_key"].str.split("|", expand=True).iloc[:, 1:3].values
bpt_dist_table["above_greenwich_benchmark"] = bpt_dist_table["R_s_baseline"] > greenwich_benchmark["R_s_baseline"]
bpt_dist_table.to_csv(TABLES_V1_DIR / "v1_bridgeport_scenario_score_distribution.csv", index=False)

by_district = bpt_dist_table.groupby("district_code")["R_s_baseline"].agg(["count", "min", "median", "max"]).reset_index()
by_district.to_csv(TABLES_V1_DIR / "v1_bridgeport_score_by_district.csv", index=False)
by_building_type = bpt_dist_table.groupby("building_form")["R_s_baseline"].agg(["count", "min", "median", "max"]).reset_index()
by_building_type.to_csv(TABLES_V1_DIR / "v1_bridgeport_score_by_building_type.csv", index=False)
print("\\nSaved: v1_greenwich_benchmark_scorecard.csv, v1_bridgeport_scenario_score_distribution.csv, v1_bridgeport_score_by_district.csv, v1_bridgeport_score_by_building_type.csv")

GREENWICH RA-4 BENCHMARK (baseline variant):
  R_s = 76.5 (coverage 90%)

BRIDGEPORT SCENARIO DISTRIBUTION (baseline variant, descriptive only -- NOT 'Bridgeport's score'):
  n = 28 scenarios
  min = 19.1, median = 27.4, max = 56.7
  mean = 31.4 (reported after min/median/max, descriptive only, unweighted)
\nBridgeport scenarios ABOVE (higher index value than) the Greenwich RA-4 benchmark: 0
Bridgeport scenarios BELOW the Greenwich RA-4 benchmark: 28
Bridgeport scenarios EQUAL to the Greenwich RA-4 benchmark: 0
\nLanguage note: 'above/below the benchmark under the V1.0 protocol' is used throughout --
never 'more/less restrictive overall,' which this scenario-level pilot cannot support as a townwide claim.
\nSaved: v1_greenwich_benchmark_scorecard.csv, v1_bridgeport_scenario_score_distribution.csv, v1_bridgeport_score_by_district.csv, v1_bridgeport_score_by_building_type.csv


## B7 (cont.) -- Sensitivity summary: does the headline comparison hold up?


In [10]:
sensitivity_summary_rows = []
for k, v in VARIANTS.items():
    gre = v[v.town == "Greenwich"]["R_s"].iloc[0]
    bpt = v[v.town == "Bridgeport"]["R_s"].dropna()
    sensitivity_summary_rows.append({
        "variant": k, "greenwich_benchmark": gre, "bridgeport_min": bpt.min(), "bridgeport_median": bpt.median(),
        "bridgeport_max": bpt.max(), "n_bridgeport_above_benchmark": int((bpt > gre).sum()),
        "n_bridgeport_below_benchmark": int((bpt < gre).sum()), "pct_bridgeport_below_benchmark": round(100 * (bpt < gre).mean(), 1),
    })
sensitivity_summary = pd.DataFrame(sensitivity_summary_rows)
sensitivity_summary.to_csv(TABLES_V1_DIR / "v1_sensitivity_summary.csv", index=False)
print(sensitivity_summary.to_string(index=False))

pct_range = sensitivity_summary["pct_bridgeport_below_benchmark"]
conclusion_stable = (pct_range.max() - pct_range.min()) < 25  # more than half the scenario distribution shifting sides would be material instability
print(f"\\nShare of Bridgeport scenarios below the Greenwich benchmark ranges {pct_range.min():.0f}%-{pct_range.max():.0f}% across all 6 variants.")
print(f"Conclusion (most Bridgeport scenarios sit below the Greenwich RA-4 benchmark under this protocol) "
      f"{'IS' if conclusion_stable else 'IS NOT'} stable across all required sensitivity variants.")

instability_flags = sensitivity_summary[sensitivity_summary["pct_bridgeport_below_benchmark"] < 50].copy()
instability_flags.to_csv(REVIEW_V1_DIR / "v1_sensitivity_instability_flags.csv", index=False)
print(f"\\nSaved: v1_sensitivity_summary.csv, v1_sensitivity_instability_flags.csv ({len(instability_flags)} variants flagged, if any, where fewer than half of Bridgeport scenarios fall below the benchmark)")

                                   variant  greenwich_benchmark  bridgeport_min  bridgeport_median  bridgeport_max  n_bridgeport_above_benchmark  n_bridgeport_below_benchmark  pct_bridgeport_below_benchmark
                                1_baseline            76.527778       19.053571          27.410714       56.660714                             0                            28                           100.0
                            2_equal_weight            78.750000       17.500000          27.500000       55.000000                             0                            28                           100.0
                      3_land_density_heavy            79.605263       23.946429          35.589286       56.339286                             0                            28                           100.0
               4_housing_entitlement_heavy            74.078947       15.482143          20.696429       63.946429                             0                            

## B6/B8 -- Coverage, exclusions, review queues, contribution summaries


In [11]:
variable_contribution = variable_scores.groupby(["town", "variable_id"])["rubric_score_0_100"].agg(["count", "mean", "min", "max"]).reset_index()
variable_contribution.to_csv(TABLES_V1_DIR / "v1_variable_contribution_summary.csv", index=False)

domain_contribution = domain_scores_df.groupby("town")[["land", "placement", "parking", "housing_entitlement", "process"]].mean().reset_index()
domain_contribution.to_csv(TABLES_V1_DIR / "v1_domain_contribution_summary.csv", index=False)

excluded_constructs = pd.DataFrame([
    {"variable": "V3", "label": "Vertical development capacity", "reason": "No common score-ready cross-town representation (Bridgeport stories-only cap vs. Greenwich documented genuine absence)"},
    {"variable": "V4", "label": "Site-intensity / ground-plane control", "reason": "No common score-ready cross-town representation (Bridgeport coverage % vs. Greenwich not-applicable)"},
    {"variable": "V10", "label": "Floor-area capacity", "reason": "Both towns independently show genuine absence of a comparable differentiating rule"},
])
excluded_constructs.to_csv(TABLES_V1_DIR / "v1_excluded_constructs_and_limitations.csv", index=False)

score_coverage = domain_scores_df[["scenario_key", "town", "coverage_fraction_baseline"]].copy()
score_coverage.to_csv(RELEASE_DIR / "comparative_score_coverage_v1.csv", index=False)

exclusion_log = freeze[~freeze.eligible_for_v1_0_freeze][["town", "scenario_key", "variable_id", "final_determination", "exclusion_reason"]]
exclusion_log.to_csv(RELEASE_DIR / "comparative_score_exclusion_log_v1.csv", index=False)
exclusion_log.to_csv(REVIEW_V1_DIR / "v1_excluded_scenarios.csv", index=False)

low_coverage = domain_scores_df[domain_scores_df.coverage_fraction_baseline < 1.0]
low_coverage.to_csv(REVIEW_V1_DIR / "v1_low_coverage_scenarios.csv", index=False)

caveated_inputs = variable_scores[variable_scores.variable_id == "V9"]
caveated_inputs.to_csv(REVIEW_V1_DIR / "v1_caveated_score_inputs.csv", index=False)

review_queue = freeze[~freeze.eligible_for_v1_0_freeze]
review_queue.to_csv(REVIEW_V1_DIR / "v1_score_input_review_queue.csv", index=False)

dependency_overlap_flags = dependency_adjustments[dependency_adjustments["shared_underlying_rule_v7_v11"]]
dependency_overlap_flags.to_csv(REVIEW_V1_DIR / "v1_dependency_overlap_flags.csv", index=False)

print(f"Saved contribution/coverage/exclusion/review outputs. Low-coverage scenarios: {len(low_coverage)}/{len(domain_scores_df)}")
print(f"Dependency-overlap (shared V7/V11 citation) flags: {len(dependency_overlap_flags)}")

Saved contribution/coverage/exclusion/review outputs. Low-coverage scenarios: 1/29
Dependency-overlap (shared V7/V11 citation) flags: 1


## Figures (consolidated, real, data-grounded set)


In [12]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5.5))
bpt_vals = domain_scores_df[domain_scores_df.town == "Bridgeport"]["R_s_baseline"].dropna()
ax.hist(bpt_vals, bins=10, color="#4C72B0", alpha=0.85, edgecolor="white", label=f"Bridgeport scenarios (n={len(bpt_vals)})")
ax.axvline(greenwich_benchmark["R_s_baseline"], color="#C44E52", linewidth=2.5, linestyle="--",
           label=f"Greenwich RA-4 benchmark ({greenwich_benchmark['R_s_baseline']:.0f})")
ax.set_xlabel("V1.0 index value (0=least restrictive, 100=most restrictive)")
ax.set_ylabel("Number of Bridgeport scenarios")
ax.set_title("V1.0 Comparative Regulation Index -- pilot\nBridgeport scenario distribution vs. Greenwich RA-4 benchmark (not a ranking)")
ax.legend()
plt.tight_layout()
fig1 = FIGURES_V1_DIR / "v1_score_distribution_bridgeport_vs_greenwich.png"
plt.savefig(fig1, dpi=150, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(10, 5.5))
domains_plot = ["land", "placement", "parking", "housing_entitlement", "process"]
gre_vals = [greenwich_benchmark[d] for d in domains_plot]
bpt_means = [domain_scores_df[domain_scores_df.town == "Bridgeport"][d].mean() for d in domains_plot]
x = np.arange(len(domains_plot)); w = 0.35
ax.bar(x - w/2, gre_vals, w, label="Greenwich RA-4", color="#C44E52")
ax.bar(x + w/2, bpt_means, w, label="Bridgeport (mean across scenarios, descriptive)", color="#4C72B0")
ax.set_xticks(x); ax.set_xticklabels([d.replace("_", "\n") for d in domains_plot])
ax.set_ylabel("Domain subscore (0-100)")
ax.set_title("Domain subscore comparison (baseline variant)")
ax.legend()
plt.tight_layout()
fig2 = FIGURES_V1_DIR / "v1_domain_subscore_comparison.png"
plt.savefig(fig2, dpi=150, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(10, 7))
heatmap_data = variable_scores.pivot_table(index="scenario_key", columns="variable_id", values="rubric_score_0_100")
im = ax.imshow(heatmap_data.values, cmap="RdYlGn_r", vmin=0, vmax=100, aspect="auto")
ax.set_yticks(range(len(heatmap_data.index))); ax.set_yticklabels(heatmap_data.index, fontsize=6)
ax.set_xticks(range(len(heatmap_data.columns))); ax.set_xticklabels(heatmap_data.columns)
ax.set_title("Variable-level score heatmap (0=least, 100=most restrictive)")
plt.colorbar(im, ax=ax, label="score")
plt.tight_layout()
fig3 = FIGURES_V1_DIR / "v1_variable_score_heatmap.png"
plt.savefig(fig3, dpi=150, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(10, 5.5))
variant_labels = list(VARIANTS.keys())
bpt_ranges = [VARIANTS[k][VARIANTS[k].town == "Bridgeport"]["R_s"].dropna() for k in variant_labels]
try:
    ax.boxplot(bpt_ranges, tick_labels=[v.replace("_", "\n") for v in variant_labels])
except TypeError:
    ax.boxplot(bpt_ranges, labels=[v.replace("_", "\n") for v in variant_labels])
for i, k in enumerate(variant_labels):
    gre_v = VARIANTS[k][VARIANTS[k].town == "Greenwich"]["R_s"].iloc[0]
    ax.scatter([i + 1], [gre_v], color="#C44E52", zorder=5, label="Greenwich RA-4" if i == 0 else None)
ax.set_ylabel("V1.0 index value")
ax.set_title("Sensitivity range across all 6 required variants")
ax.legend()
plt.xticks(fontsize=7)
plt.tight_layout()
fig4 = FIGURES_V1_DIR / "v1_sensitivity_range_plot.png"
plt.savefig(fig4, dpi=150, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(domain_scores_df["scenario_key"].where(domain_scores_df.town == "Bridgeport").dropna(),
       domain_scores_df.loc[domain_scores_df.town == "Bridgeport", "coverage_fraction_baseline"], color="#55A868")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Coverage fraction (domain weight actually used)")
ax.set_title("Score coverage by Bridgeport scenario (baseline variant)")
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
fig5 = FIGURES_V1_DIR / "v1_score_coverage_plot.png"
plt.savefig(fig5, dpi=150, bbox_inches="tight"); plt.close()

figure_paths = [fig1, fig2, fig3, fig4, fig5]
for p in figure_paths:
    assert p.exists() and p.stat().st_size > 0
print(f"All {len(figure_paths)} figures rendered and nonempty.")
print("Consolidated from the originating spec's 9 named figures + evidence-card library into 5 real, data-grounded figures;")
print("the remaining named figures (eligibility flow, dependency-adjustment visual, methodology flow, excluded-constructs panel,")
print("evidence cards) are deferred -- see KNOWN_LIMITATIONS.md -- rather than produced as thin/placeholder images.")

All 5 figures rendered and nonempty.
Consolidated from the originating spec's 9 named figures + evidence-card library into 5 real, data-grounded figures;
the remaining named figures (eligibility flow, dependency-adjustment visual, methodology flow, excluded-constructs panel,
evidence cards) are deferred -- see KNOWN_LIMITATIONS.md -- rather than produced as thin/placeholder images.


## Scenario scores (final consolidated output) and methodology manifest


In [13]:
scenario_scores_final = domain_scores_df.merge(
    sensitivity_long.pivot_table(index="scenario_key", columns="variant", values="R_s"), on="scenario_key"
)
scenario_scores_final.to_csv(RELEASE_DIR / "comparative_scenario_scores_v1.csv", index=False)
scenario_scores_final.to_parquet(RELEASE_DIR / "comparative_scenario_scores_v1.parquet", index=False)
print(f"Saved: comparative_scenario_scores_v1.csv / .parquet ({len(scenario_scores_final)} scenarios x all 6 variants)")

methodology = {
    "release_version": RELEASE_VERSION, "timestamp": RUN_TIMESTAMP,
    "frozen_input_hash": FROZEN_HASH_NOW,
    "unit_of_analysis": "scenario = <town, district, building_form/use, site/frontage condition>",
    "score_direction": "0=least restrictive, 100=most restrictive",
    "domain_formula": {
        "L_s": "0.30*V1 + 0.25*V2 + 0.45*V9 (renormalized over available components)",
        "B_s": "V5 (conventional_setback or build_to_flexibility sub-rubric)",
        "P_s": "V6", "H_s": "0.45*V7 + 0.25*V8 + 0.30*V12 (gated off entirely where V7=not_applicable)",
        "D_s": "V11 (halved when shared_underlying_rule=true with V7)",
        "R_s": "0.25*L + 0.15*B + 0.10*P + 0.35*H + 0.15*D (renormalized over available domains)",
    },
    "weights_are_theory_driven_not_empirical": True,
    "dependency_policy": "V7/V11 shared-citation check computed per scenario from actual source_page/section match; V9 caveat carried through and tested in sensitivity variant 6",
    "sensitivity_variants_run": list(VARIANTS.keys()),
    "greenwich_benchmark_R_s_baseline": float(greenwich_benchmark["R_s_baseline"]),
    "bridgeport_scenario_count": int(len(bpt_dist)),
    "bridgeport_baseline_distribution": {"min": float(bpt_dist.min()), "median": float(bpt_dist.median()), "max": float(bpt_dist.max())},
    "pct_bridgeport_below_benchmark_range_across_variants": [float(pct_range.min()), float(pct_range.max())],
    "excluded_context_variables": ["V3", "V4", "V10"],
    "no_townwide_ranking_statement": "No single Bridgeport townwide score was computed. No ranking, dominant district, statewide percentile, or causal claim is made anywhere in this notebook.",
}
(RELEASE_DIR / "comparative_score_methodology_v1.json").write_text(json.dumps(methodology, indent=2))
print(json.dumps(methodology, indent=2))

Saved: comparative_scenario_scores_v1.csv / .parquet (29 scenarios x all 6 variants)
{
  "release_version": "v1.0.0-pilot",
  "timestamp": "2026-10-01T21:48:09.634144+00:00",
  "frozen_input_hash": "97969cdc26e6344841fea9fb9b2d80bc322b8c1daadb06c599ce91037daa042e",
  "unit_of_analysis": "scenario = <town, district, building_form/use, site/frontage condition>",
  "score_direction": "0=least restrictive, 100=most restrictive",
  "domain_formula": {
    "L_s": "0.30*V1 + 0.25*V2 + 0.45*V9 (renormalized over available components)",
    "B_s": "V5 (conventional_setback or build_to_flexibility sub-rubric)",
    "P_s": "V6",
    "H_s": "0.45*V7 + 0.25*V8 + 0.30*V12 (gated off entirely where V7=not_applicable)",
    "D_s": "V11 (halved when shared_underlying_rule=true with V7)",
    "R_s": "0.25*L + 0.15*B + 0.10*P + 0.35*H + 0.15*D (renormalized over available domains)"
  },
  "weights_are_theory_driven_not_empirical": true,
  "dependency_policy": "V7/V11 shared-citation check computed per sc

## Markdown comparison summary


In [14]:
comparison_md = f'''# v1_bridgeport_greenwich_comparison.md

**Important: this is a source-linked, scenario-aware pilot comparison. It does not constitute a
townwide ranking, statewide ranking, or causal claim.**

## Greenwich RA-4 benchmark

R_s = {greenwich_benchmark["R_s_baseline"]:.1f} / 100 (baseline variant, {greenwich_benchmark["coverage_fraction_baseline"]:.0%} domain coverage)

## Bridgeport scenario distribution ({len(bpt_dist)} eligible scenarios)

- min = {bpt_dist.min():.1f}, median = {bpt_dist.median():.1f}, max = {bpt_dist.max():.1f}
- {n_below} of {len(bpt_dist)} scenarios ({100*n_below/len(bpt_dist):.0f}%) fall below the Greenwich RA-4 benchmark
- {n_above} of {len(bpt_dist)} scenarios ({100*n_above/len(bpt_dist):.0f}%) fall above it

## Sensitivity

Across all 6 required variants, the share of Bridgeport scenarios below the Greenwich benchmark
ranges {pct_range.min():.0f}%-{pct_range.max():.0f}%. See `v1_sensitivity_summary.csv` for every variant.

## What this does NOT mean

This does not mean "Bridgeport is less restrictive than Greenwich" as towns. It means: under this
V1.0 pilot protocol's nine scored variables and declared weights, most of Bridgeport's eligible
*building-form scenarios* produce a lower index value than Greenwich's single documented, entirely
single-family-exclusive RA-4 district. V3 (height), V4 (site intensity), and V10 (floor area) are
excluded from this index and are visible separately as context.
'''
(TABLES_V1_DIR / "v1_bridgeport_greenwich_comparison.md").write_text(comparison_md)
print("Saved: v1_bridgeport_greenwich_comparison.md")
print(comparison_md)

Saved: v1_bridgeport_greenwich_comparison.md
# v1_bridgeport_greenwich_comparison.md

**Important: this is a source-linked, scenario-aware pilot comparison. It does not constitute a
townwide ranking, statewide ranking, or causal claim.**

## Greenwich RA-4 benchmark

R_s = 76.5 / 100 (baseline variant, 90% domain coverage)

## Bridgeport scenario distribution (28 eligible scenarios)

- min = 19.1, median = 27.4, max = 56.7
- 28 of 28 scenarios (100%) fall below the Greenwich RA-4 benchmark
- 0 of 28 scenarios (0%) fall above it

## Sensitivity

Across all 6 required variants, the share of Bridgeport scenarios below the Greenwich benchmark
ranges 100%-100%. See `v1_sensitivity_summary.csv` for every variant.

## What this does NOT mean

This does not mean "Bridgeport is less restrictive than Greenwich" as towns. It means: under this
V1.0 pilot protocol's nine scored variables and declared weights, most of Bridgeport's eligible
*building-form scenarios* produce a lower index value than G

## Final manifest and validation


In [15]:
score_manifest = {
    "release_version": RELEASE_VERSION, "timestamp": RUN_TIMESTAMP,
    "freeze_file_hash": FROZEN_HASH_NOW,
    "rubric_file_hash": hashlib.sha256((RELEASE_DIR / "comparative_scoring_rubrics_v1.csv").read_bytes()).hexdigest(),
    "formulas": methodology["domain_formula"], "weights": DOMAIN_WEIGHTS_BASELINE,
    "dependency_policy": methodology["dependency_policy"],
    "scenario_counts": {"greenwich": 1, "bridgeport_eligible": int(len(bpt_dist))},
    "inclusion_exclusion_counts": {"frozen_rows_eligible": int(freeze.eligible_for_v1_0_freeze.sum()), "frozen_rows_excluded": int((~freeze.eligible_for_v1_0_freeze).sum())},
    "coverage_statistics": {"mean_coverage_fraction": float(domain_scores_df["coverage_fraction_baseline"].mean())},
    "sensitivity_models": list(VARIANTS.keys()),
    "quality_flags": {"low_coverage_scenarios": int(len(low_coverage)), "dependency_overlap_flags": int(len(dependency_overlap_flags))},
    "limitations": [
        "Greenwich V6 not independently scored this pass (see rubric limitations field)",
        "V9 is a caveated variable; sensitivity variant 6 reports the conservative exclusion",
        "This is a pilot: nine of twelve registry variables are scored; three (V3/V4/V10) are excluded context",
    ],
    "no_townwide_ranking_statement": "No townwide ranking, dominant-district selection, statewide percentile, or causal conclusion was produced.",
}
score_manifest_path = LOGS_DIR / "comparative_residential_regulation_index_v1_manifest.json"
score_manifest_path.write_text(json.dumps(score_manifest, indent=2, default=str))
print(f"Saved: {score_manifest_path.relative_to(ROOT)}")

post_hash = hashlib.sha256(FREEZE_PATH.read_bytes()).hexdigest()
assert post_hash == FROZEN_HASH_NOW, "frozen scoring input must remain unchanged after scoring"
print(f"\\nFrozen input file confirmed unchanged after scoring: {post_hash == FROZEN_HASH_NOW}")
print("\\nNotebook 15 complete.")

Saved: outputs/logs/comparative_residential_regulation_index_v1_manifest.json
\nFrozen input file confirmed unchanged after scoring: True
\nNotebook 15 complete.
